# Feature selection and interpretation

Compare L1 sparsity, mutual information, RFE and tree-based selection. Then use held-out permutation importance and optional SHAP for nonlinear models.

In [ ]:
from pathlib import Path

import pandas as pd

from scania_aps.data import TEST_FILENAME, TRAIN_FILENAME, read_raw_csv

ROOT = Path.cwd().resolve()
if ROOT.name == "experiments":
    ROOT = ROOT.parent
TRAIN = ROOT / "data" / "raw" / TRAIN_FILENAME
TEST = ROOT / "data" / "raw" / TEST_FILENAME
ARTIFACTS = ROOT / "artifacts"
assert TRAIN.exists() and TEST.exists(), "Run: poetry run scania-aps download"
train = read_raw_csv(TRAIN)
test = read_raw_csv(TEST)
print(train.X.shape, test.X.shape, train.y.mean(), test.y.mean())

In [ ]:
from scania_aps.studies import run_feature_selection_study

run_feature_selection_study(TRAIN, TEST, ARTIFACTS)

In [ ]:
# Optional interpretability example after a model-study run
import joblib

from scania_aps.feature_selection import permutation_ranking

model_path = ARTIFACTS / "model_study" / "xgboost_model.joblib"
if model_path.exists():
    model = joblib.load(model_path)
    ranking = permutation_ranking(model, test.X, test.y, n_repeats=5)
    display(pd.DataFrame([x.__dict__ for x in ranking]).head(30))
    # SHAP can be expensive; cap the sample deliberately.
    # explanation = shap_values(model, test.X, max_rows=500)